We will run the notebook at root level.

In [5]:
import sys
from pathlib import Path

# notebook is one level below root
ROOT = Path.cwd().parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


# Run on pilot sample

In [10]:
from pathlib import Path
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
import json


ROOT = Path(r"c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow")
PILOT = ROOT / "Pilot_Evaluation"

# Setup paths for pilot study
PILOT_VECTOR_STORE = PILOT /"pilot_study_10_vectors"
PILOT_DATA_FOLDER = PILOT /"DATA_sample_10"
RESULTS_FOLDER = PILOT /"pilot_study_results"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

# Get list of papers from pilot data
pilot_papers = sorted([p for p in PILOT_DATA_FOLDER.glob("*.pdf")])

print(f"📚 Found {len(pilot_papers)} papers in pilot study")
print(f"📁 Vector store: {PILOT_VECTOR_STORE.absolute()}")
print(f"📁 Results will be stored in: {RESULTS_FOLDER.absolute()}\n")

# Load vectorstore with pilot study embeddings
vectorstore = Chroma(
    collection_name="pilot_study_10",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=str(PILOT_VECTOR_STORE)
)

print(f"✅ Vectorstore loaded with {len(pilot_papers)} papers' embeddings")

📚 Found 10 papers in pilot study
📁 Vector store: c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\pilot_study_10_vectors
📁 Results will be stored in: c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\pilot_study_results

✅ Vectorstore loaded with 10 papers' embeddings


In [11]:
def create_paper_result_structure(paper_id):
    """Create organized folder structure for each paper's results."""
    paper_results_dir = RESULTS_FOLDER / paper_id
    paper_results_dir.mkdir(parents=True, exist_ok=True)
    
    # Create subdirectories for different output types
    subfolders = ["raw_outputs", "aggregated", "logs"]
    for subfolder in subfolders:
        (paper_results_dir / subfolder).mkdir(exist_ok=True)
    
    return paper_results_dir

def save_paper_results(paper_id, result_state):
    """Save all DSRP outputs and state for a paper."""
    paper_dir = create_paper_result_structure(paper_id)
    
    # Save raw DSRP outputs per node
    dsrp_outputs = result_state.get("dsrp_outputs", {})
    for node_name, node_output in dsrp_outputs.items():
        node_file = paper_dir / "raw_outputs" / f"{node_name}.json"
        with open(node_file, 'w', encoding='utf-8') as f:
            json.dump(node_output, f, indent=2, ensure_ascii=False)
    
    # Save aggregated results
    aggregated = {
        "paper_id": paper_id,
        "gatekeeper_decision": result_state.get("gatekeeper", {}).get("final_classification"),
        "dsrp_outputs": dsrp_outputs
    }
    
    agg_file = paper_dir / "aggregated" / "results.json"
    with open(agg_file, 'w', encoding='utf-8') as f:
        json.dump(aggregated, f, indent=2, ensure_ascii=False)
    
    return paper_dir

print("✅ Result storage functions defined")

✅ Result storage functions defined


In [12]:
# Custom Define Papers 
pilot_papers = ["2020 - 8"]

In [13]:
pilot_papers = [Path(p) for p in pilot_papers]


In [14]:

from graph import semi_parallel_graph
# or: from graph.semi_parallel_graph import <name>


Current working directory: c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow


In [15]:
graph = semi_parallel_graph.graph


In [16]:
from utils.dsrp_state import DSRPState

In [19]:
# Run agentic workflow on all pilot papers
import time
from datetime import datetime

execution_log = {
    "start_time": datetime.now().isoformat(),
    "papers_processed": [],
    "errors": []
}

print("=" * 80)
print(f"🚀 Starting Agentic Workflow on {len(pilot_papers)} Papers")
print("=" * 80 + "\n")

for idx, paper_path in enumerate(pilot_papers, 1):
    paper_name = paper_path.stem  # filename without extension
    
    print(f"[{idx}/{len(pilot_papers)}] Processing: {paper_path.name}")
    print("-" * 80)
    
    try:
        start_time = time.time()
        
        # Initialize state with paper and vectorstore info
        paper_state: DSRPState = {
            "paper_id": paper_name,
            "gatekeeper": {},
            "dsrp_outputs": {},
            "collection_name": "pilot_study_10",
            "persist_directory": str(PILOT_VECTOR_STORE),
            "embedding_model": "text-embedding-3-small"
        }
        
        # Run the agentic workflow
        result = graph.invoke(paper_state)
        
        # Save results to organized structure
        paper_dir = save_paper_results(paper_name, result)
        
        elapsed = time.time() - start_time
        
        # Log execution details
        gatekeeper_status = result.get("gatekeeper", {}).get("final_classification", "Unknown")
        execution_log["papers_processed"].append({
            "paper_id": paper_name,
            "status": "success",
            "gatekeeper_decision": gatekeeper_status,
            "results_path": str(paper_dir),
            "execution_time_seconds": round(elapsed, 2)
        })
        
        print(f"✅ Completed ({elapsed:.2f}s)")
        print(f"   📊 Gatekeeper: {gatekeeper_status}")
        print(f"   📁 Results saved to: {paper_dir.resolve().relative_to(Path.cwd().resolve())}\n")
        
    except Exception as e:
        print(f"❌ Error processing {paper_path.name}: {str(e)}\n")
        execution_log["errors"].append({
            "paper_id": paper_name,
            "error": str(e)
        })

execution_log["end_time"] = datetime.now().isoformat()

# Save execution log
log_file = RESULTS_FOLDER / "execution_log.json"
with open(log_file, 'w', encoding='utf-8') as f:
    json.dump(execution_log, f, indent=2, ensure_ascii=False)

print("=" * 80)
print(f"✅ Workflow Complete!")
print(f"📊 Successfully processed: {len(execution_log['papers_processed'])}/{len(pilot_papers)} papers")
print(f"⚠️  Errors: {len(execution_log['errors'])}")
print(f"📁 Results location: {RESULTS_FOLDER.absolute()}")
print("=" * 80)

🚀 Starting Agentic Workflow on 1 Papers

[1/1] Processing: 2020 - 8
--------------------------------------------------------------------------------
✅ Completed (182.05s)
   📊 Gatekeeper: Include
   📁 Results saved to: Pilot_Evaluation\pilot_study_results\2020 - 8

✅ Workflow Complete!
📊 Successfully processed: 1/1 papers
⚠️  Errors: 0
📁 Results location: c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\pilot_study_results


In [20]:
import pandas as pd
from collections import Counter

# Load execution log
with open(RESULTS_FOLDER / "execution_log.json", 'r', encoding='utf-8') as f:
    log = json.load(f)

# Create summary report
summary_df = pd.DataFrame(log["papers_processed"])

print("\n📋 PILOT STUDY WORKFLOW SUMMARY\n")
print(summary_df[["paper_id", "gatekeeper_decision", "execution_time_seconds"]].to_string(index=False))

# Gatekeeper decision breakdown
if not summary_df.empty:
    decision_counts = summary_df["gatekeeper_decision"].value_counts()
    print("\n\n📊 Gatekeeper Decisions:")
    for decision, count in decision_counts.items():
        print(f"   {decision}: {count} papers")

# Execution statistics
total_time = sum(p.get("execution_time_seconds", 0) for p in log["papers_processed"])
print(f"\n⏱️  Total Execution Time: {total_time:.2f} seconds")
print(f"⏱️  Average per Paper: {total_time/len(log['papers_processed']):.2f} seconds")

# Error summary
if log["errors"]:
    print(f"\n⚠️  Errors ({len(log['errors'])}):")
    for error in log["errors"]:
        print(f"   - {error['paper_id']}: {error['error'][:50]}...")
else:
    print(f"\n✅ No errors encountered")

print(f"\n📁 All results organized in: {RESULTS_FOLDER.resolve().relative_to(Path.cwd().resolve())}/")
print(f"   Each paper has its own folder with:")
print(f"   - raw_outputs/: Individual node results")
print(f"   - aggregated/: Compiled results for paper")
print(f"   - logs/: Processing logs")


📋 PILOT STUDY WORKFLOW SUMMARY

paper_id gatekeeper_decision  execution_time_seconds
2020 - 8             Include                  182.05


📊 Gatekeeper Decisions:
   Include: 1 papers

⏱️  Total Execution Time: 182.05 seconds
⏱️  Average per Paper: 182.05 seconds

✅ No errors encountered

📁 All results organized in: Pilot_Evaluation\pilot_study_results/
   Each paper has its own folder with:
   - raw_outputs/: Individual node results
   - aggregated/: Compiled results for paper
   - logs/: Processing logs
